# Q1

## Q1 code

In [18]:
spark.stop()
import os
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"


In [29]:
# import
import pyspark
from pyspark.sql import SparkSession, SQLContext
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import *
from pyspark.sql.types import *
import numpy as np

# column config
col_names = ["duration","protocol_type","service","flag","src_bytes",
"dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
"logged_in","num_compromised","root_shell","su_attempted","num_root",
"num_file_creations","num_shells","num_access_files","num_outbound_cmds",
"is_host_login","is_guest_login","count","srv_count","serror_rate",
"srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
"diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
"dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
"dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
"dst_host_rerror_rate","dst_host_srv_rerror_rate","class","difficulty"]

nominal_cols = ['protocol_type','service','flag']
binary_cols = ['land', 'logged_in', 'root_shell', 'su_attempted', 'is_host_login','is_guest_login']
continuous_cols = ['duration' ,'src_bytes', 'dst_bytes', 'wrong_fragment','urgent', 'hot',
'num_failed_logins', 'num_compromised', 'num_root' ,'num_file_creations',
'num_shells', 'num_access_files', 'num_outbound_cmds', 'count' ,'srv_count',
'serror_rate', 'srv_serror_rate' ,'rerror_rate' ,'srv_rerror_rate',
'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate' ,'dst_host_count',
'dst_host_srv_count' ,'dst_host_same_srv_rate' ,'dst_host_diff_srv_rate',
'dst_host_same_src_port_rate' ,'dst_host_srv_diff_host_rate',
'dst_host_serror_rate' ,'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
'dst_host_srv_rerror_rate']

# transformer
class OutcomeCreater(Transformer):
    def __init__(self):
        super().__init__()
    def _transform(self, dataset):
        label_to_binary = udf(lambda name: 0.0 if name == 'normal' else 1.0)
        output_df = dataset.withColumn('outcome', label_to_binary(col('class'))).drop("class")
        output_df = output_df.withColumn('outcome', col('outcome').cast(DoubleType()))
        output_df = output_df.drop('difficulty')
        return output_df

class FeatureTypeCaster(Transformer):
    def __init__(self):
        super().__init__()
    def _transform(self, dataset):
        output_df = dataset
        for c in binary_cols + continuous_cols:
            output_df = output_df.withColumn(c, col(c).cast(DoubleType()))
        return output_df

class ColumnDropper(Transformer):
    def __init__(self, columns_to_drop=None):
        super().__init__()
        self.columns_to_drop = columns_to_drop or []
    def _transform(self, dataset):
        output_df = dataset
        for c in self.columns_to_drop:
            output_df = output_df.drop(c)
        return output_df
def get_preprocess_pipeline():
    stage_typecaster = FeatureTypeCaster()
    nominal_id_cols = [x + "_index" for x in nominal_cols]
    nominal_onehot_cols = [x + "_encoded" for x in nominal_cols]
    stage_nominal_indexer = StringIndexer(inputCols=nominal_cols, outputCols=nominal_id_cols, handleInvalid="keep")
    stage_nominal_onehot_encoder = OneHotEncoder(inputCols=nominal_id_cols, outputCols=nominal_onehot_cols, handleInvalid="keep")

    feature_cols = continuous_cols + binary_cols + nominal_onehot_cols
    corelated_cols_to_remove = ["dst_host_serror_rate","srv_serror_rate","dst_host_srv_serror_rate",
                                "srv_rerror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate"]
    for c in corelated_cols_to_remove:
        if c in feature_cols:
            feature_cols.remove(c)

    stage_vector_assembler = VectorAssembler(inputCols=feature_cols, outputCol="vectorized_features", handleInvalid="keep")
    stage_scaler = StandardScaler(inputCol='vectorized_features', outputCol='features')

    stage_outcome = OutcomeCreater()

    stage_column_dropper = ColumnDropper(columns_to_drop=nominal_cols + nominal_id_cols + nominal_onehot_cols +
                                         binary_cols + continuous_cols + ['vectorized_features'])
    lr = LogisticRegression(featuresCol='features', labelCol='outcome')
    


    pipeline = Pipeline(stages=[
        stage_typecaster,
        stage_nominal_indexer,
        stage_nominal_onehot_encoder,
        stage_vector_assembler,
        stage_scaler,
        stage_outcome,
        stage_column_dropper,
        lr
    ])
    return pipeline

# load the training and test dataframe 
spark = SparkSession.builder.master("local[*]").appName("Q1").getOrCreate()

pipeline = get_preprocess_pipeline() 

# read RAW train/test
train_raw = spark.read.csv('KDDTrain+.txt', header=False).toDF(*col_names)
test_raw  = spark.read.csv('KDDTest+.txt',  header=False).toDF(*col_names)

# tune the hyper-parameters
lr = [s for s in pipeline.getStages() if isinstance(s, LogisticRegression)][0]

paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.0, 0.01, 0.1])
             .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
             .addGrid(lr.maxIter, [50,100])
             .build())

evaluator = BinaryClassificationEvaluator(labelCol='outcome', metricName='areaUnderROC')

cv = CrossValidator(estimator=pipeline,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=3,
                    parallelism=2)

# print the stage of pipline
print("PIPELINE STAGES:", [type(s).__name__ for s in pipeline.getStages()])

# fit on raw train
cv_model = cv.fit(train_raw)
test_pred = cv_model.transform(test_raw)

# Print schema and first few rows
test_pred.printSchema()
test_pred.select("features", "outcome", "probability", "prediction").show(5, truncate=False)

PIPELINE STAGES: ['FeatureTypeCaster', 'StringIndexer', 'OneHotEncoder', 'VectorAssembler', 'StandardScaler', 'OutcomeCreater', 'ColumnDropper', 'LogisticRegression']
root
 |-- features: vector (nullable = true)
 |-- outcome: double (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+------------------------------------------+----------+
|features                                                                                                                                                                                     

## Q1 output

Refer to q1.png

# Q2

## Q2-1

# Reference: 
1.for attack types: https://www.researchgate.net/figure/Attack-types-of-DoS-R2L-U2R-Probe-categories_tbl1_327110465


In [21]:
from pyspark.ml.feature import StringIndexer

train_raw = spark.read.csv('KDDTrain+.txt', header=False).toDF(*col_names)
test_raw = spark.read.csv('KDDTest+.txt',  header=False).toDF(*col_names)

DOS=['back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','mailbomb']
probing=['ipsweep','nmap','portsweep','satan','mscan','saint']
U2R=['buffer_overflow','loadmodule','perl','rootkit','ps','sqlattack','xterm']

def add_attack(df):
    c = lower(col("class"))
    return (df.withColumn(
              "attack",
              when(c=='normal','normal')
              .when(c.isin(DOS),   'DOS')
              .when(c.isin(probing), 'probing')
              .when(c.isin(U2R),   'U2R')
              .otherwise('R2L'))
           )

train_raw2 = add_attack(train_raw)
test_raw2 = add_attack(test_raw)
label_indexer = StringIndexer(inputCol="attack", outputCol="label", handleInvalid="keep")
label_model = label_indexer.fit(train_raw2)
train_idx = label_model.transform(train_raw2)
test_idx = label_model.transform(test_raw2)
def get_preprocess_pipeline():
    stage_typecaster = FeatureTypeCaster()
    nominal_id_cols = [x + "_index" for x in nominal_cols]
    nominal_oh_cols = [x + "_encoded" for x in nominal_cols]
    stage_nominal_indexer = StringIndexer(inputCols=nominal_cols, outputCols=nominal_id_cols, handleInvalid="keep")
    stage_nominal_onehot = OneHotEncoder(inputCols=nominal_id_cols, outputCols=nominal_oh_cols, handleInvalid="keep")
    feature_cols = continuous_cols + binary_cols + nominal_oh_cols
    corelated_cols_to_remove = ["dst_host_serror_rate","srv_serror_rate","dst_host_srv_serror_rate",
                                "srv_rerror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate"]
    feature_cols= [c for c in feature_cols if c not in corelated_cols_to_remove]
    stage_vec= VectorAssembler(inputCols=feature_cols, outputCol="vectorized_features", handleInvalid="keep")
    stage_scaler= StandardScaler(inputCol="vectorized_features", outputCol="features")
    stage_drop= ColumnDropper(columns_to_drop = nominal_cols + nominal_id_cols + nominal_oh_cols +
                                 binary_cols + continuous_cols + ['vectorized_features','difficulty'])
    return Pipeline(stages=[stage_typecaster, stage_nominal_indexer, stage_nominal_onehot,
                            stage_vec, stage_scaler, stage_drop])
feat_pipe = get_preprocess_pipeline()
feat_model = feat_pipe.fit(train_idx)
train_df = feat_model.transform(train_idx)
test_df = feat_model.transform(test_idx)

train_df.printSchema()
train_df.select("label","attack").show(20, truncate=False)


root
 |-- class: string (nullable = true)
 |-- attack: string (nullable = false)
 |-- label: double (nullable = false)
 |-- features: vector (nullable = true)

+-----+-------+
|label|attack |
+-----+-------+
|0.0  |normal |
|0.0  |normal |
|1.0  |DOS    |
|0.0  |normal |
|0.0  |normal |
|1.0  |DOS    |
|1.0  |DOS    |
|1.0  |DOS    |
|1.0  |DOS    |
|1.0  |DOS    |
|1.0  |DOS    |
|1.0  |DOS    |
|0.0  |normal |
|3.0  |R2L    |
|1.0  |DOS    |
|1.0  |DOS    |
|0.0  |normal |
|2.0  |probing|
|0.0  |normal |
|0.0  |normal |
+-----+-------+
only showing top 20 rows



## Q2-2

In [26]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import IndexToString, StringIndexerModel
from pyspark.sql import functions as F

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)

# Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train_df)
predictions_train_lr = lr_model.transform(train_df)
predictions_test_lr = lr_model.transform(test_df)

# Train accuracy
accuracy_train = (
    predictions_train_lr.filter(F.col("label") == F.col("prediction"))
    .count()
    / float(predictions_train_lr.count())
)

# Test accuracy
accuracy_test = (
    predictions_test_lr.filter(F.col("label") == F.col("prediction"))
    .count()
    / float(predictions_test_lr.count())
)

print(f"[Logistic Regression] Train Accuracy : {np.round(accuracy_train * 100, 2)}%")
print(f"[Logistic Regression] Test Accuracy  : {np.round(accuracy_test * 100, 2)}%")

[Logistic Regression] Train Accuracy : 98.77%
[Logistic Regression] Test Accuracy  : 75.89%


In [27]:
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50)
rf_model = rf.fit(train_df)
predictions_train_rf = rf_model.transform(train_df)
predictions_test_rf  = rf_model.transform(test_df)

accuracy_train_rf = (predictions_train_rf.filter(F.col("label")==F.col("prediction")).count()
                / float(predictions_train_rf.count()))
accuracy_test_rf  = (predictions_test_rf.filter(F.col("label")==F.col("prediction")).count()
                / float(predictions_test_rf.count()))

print(f"[Random Forest] Train Accuracy : {np.round(accuracy_train_rf*100,2)}%")
print(f"[Random Forest] Test  Accuracy : {np.round(accuracy_test_rf*100,2)}%")

[Random Forest] Train Accuracy : 97.19%
[Random Forest] Test  Accuracy : 72.79%


In [24]:
lr_pred_test = lr_model.transform(test_df)
rf_pred_test = rf_model.transform(test_df)
def confusion_matrix_named(pred_df: DataFrame, labels: list, true_col="attack"):
    to_pred = IndexToString(inputCol="prediction", outputCol="pred_attack", labels=labels)
    named = to_pred.transform(pred_df)
    return (named.groupBy(true_col, "pred_attack").count()
                 .groupBy(true_col)
                 .pivot("pred_attack", labels)
                 .sum("count")
                 .na.fill(0)
                 .orderBy(true_col))

labels_list = label_model.labels

cm_lr = confusion_matrix_named(lr_pred_test, labels_list)
cm_rf = confusion_matrix_named(rf_pred_test, labels_list)

print("LR Confusion Matrix (attack names):")
cm_lr.show(truncate=False)
print("RF Confusion Matrix (attack names):")
cm_rf.show(truncate=False)

LR Confusion Matrix (attack names):
+-------+------+----+-------+---+---+
|attack |normal|DOS |probing|R2L|U2R|
+-------+------+----+-------+---+---+
|DOS    |1203  |6244|11     |0  |0  |
|R2L    |2749  |3   |3      |129|3  |
|U2R    |44    |2   |0      |4  |17 |
|normal |9037  |421 |249    |0  |4  |
|probing|498   |135 |1681   |106|1  |
+-------+------+----+-------+---+---+

RF Confusion Matrix (attack names):
+-------+------+----+-------+---+---+
|attack |normal|DOS |probing|R2L|U2R|
+-------+------+----+-------+---+---+
|DOS    |1953  |5500|5      |0  |0  |
|R2L    |2868  |16  |3      |0  |0  |
|U2R    |67    |0   |0      |0  |0  |
|normal |9496  |37  |178    |0  |0  |
|probing|853   |155 |1413   |0  |0  |
+-------+------+----+-------+---+---+



## Q2-3

In [28]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
import numpy as np

# Logistic Regression tuning
lr = LogisticRegression(featuresCol='features', labelCol='label')

# paramGrid for lr model
lr_paramGrid = (ParamGridBuilder()
              .addGrid(lr.regParam, [1e-5,1e-4,1e-3,1e-2])
              .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
              .addGrid(lr.maxIter, [100])
              .build())

# CrossValidator for lr
lr_cv = CrossValidator(estimator=lr,
                       estimatorParamMaps=lr_paramGrid,
                       evaluator=evaluator_acc,  
                       numFolds=5,
                       parallelism=2)

# Using lr_cv to fit the train_df(features+label)
lr_cv_model = lr_cv.fit(train_df)


# Random Forest tuning
rf = RandomForestClassifier(featuresCol='features', labelCol='label')

# paramGrid for rf model
rf_paramGrid = (ParamGridBuilder()
              .addGrid(rf.maxDepth, [5,10,20])
              .addGrid(rf.numTrees, [10,20,50])
              .build())

# CrossValidator for rf
rf_cv = CrossValidator(estimator=rf,
                       estimatorParamMaps=rf_paramGrid,
                       evaluator=evaluator_acc,
                       numFolds=3,
                       parallelism=2)

# Using rf_cv to fit the train_df(features+label)
rf_cv_model = rf_cv.fit(train_df)

# Calculate the accuracy of the tuned model
lr_tuned_predictions = lr_cv_model.transform(test_df)
rf_tuned_predictions = rf_cv_model.transform(test_df)

lr_tuned_accuracy=evaluator_acc.evaluate(lr_tuned_predictions)
rf_tuned_accuracy=evaluator_acc.evaluate(rf_tuned_predictions)

print(f"[Logistic Regression]:{np.round(lr_tuned_accuracy * 100, 2)}%")
print(f"[Random Forest]:{np.round(rf_tuned_accuracy * 100, 2)}%")

best_lr = lr_cv_model.bestModel
best_rf = rf_cv_model.bestModel

print(f"LR: regParam={best_lr.getRegParam()}, elasticNetParam={best_lr.getElasticNetParam()}")
print(f"RF: maxDepth={best_rf.getMaxDepth()}, numTrees={best_rf.getNumTrees}")

[Logistic Regression]:74.21%
[Random Forest]:74.67%
LR: regParam=1e-05, elasticNetParam=1.0
RF: maxDepth=20, numTrees=50


## Q2-4

The reason why i chose Logistic Regression and Random Forest as the two ML model is that they represent two levels of complexity and also learning behavior.

LR(as what we have learned from slides) is the simplest ML learning model. It will not overfit and serve as a good baseline for multi-class classification work.

RF is under the category of Decision Tree, which is highly suitable for classification, since I converted the category attack into numeric labels(0,1,2,3,4) in this pipeline.

For Logistic Regression, I tuned the hyper-parameters regParam, elasticNetParam, and maxIter:
regParam(regularization parameter) controls the strength of regularization and prevents overfitting.
elasticNetParam adjusts the balance between L1 and L2 regularization.
maxIter(maximum number ofiterations) ensures the optimization process converges.
The parameter grid was designed with small to moderate ranges [1e-5,1e-4,1e-3,1e-2] for regParam, [0.0, 0.5, 1.0] for elasticNetParam) to test both weak and strong regularization while keeping computation efficient.

For Random Forest, I tuned the number of trees (numTrees) and the maximum tree depth (maxDepth):
Increasing numTrees generally improves performance but increases computation cost.

maxDepth controls how complex each tree can become, balancing bias and variance.
The grid (numTrees = [10,20,50], maxDepth = [5,10,20]) was designed to observe their impact on accuracy.

Comparsion:
Although the tuned Logistic Regression model had slightly lower test accuracy (74.21%) compared to its untuned version (75.89%), this result may indicate overfitting during cross-validation.
The model likely fit the training folds too tightly due to the very small regParam and a fully L1-based penalty
In contrast, the tuned Random Forest model maintained a better test performance (74.67%), suggesting it generalizes more consistently despite its larger capacity.

## Q2-5

### Q2-5 code

In [ ]:
import pyspark
from pyspark.sql import SparkSession, SQLContext
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import *
from pyspark.sql.types import *
import numpy as np

spark = SparkSession.builder.master("yarn").appName("Q2-5").getOrCreate()

col_names = ["duration","protocol_type","service","flag","src_bytes",
"dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
"logged_in","num_compromised","root_shell","su_attempted","num_root",
"num_file_creations","num_shells","num_access_files","num_outbound_cmds",
"is_host_login","is_guest_login","count","srv_count","serror_rate",
"srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
"diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
"dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
"dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
"dst_host_rerror_rate","dst_host_srv_rerror_rate","class","difficulty"]

nominal_cols = ['protocol_type','service','flag']
binary_cols = ['land', 'logged_in', 'root_shell', 'su_attempted', 'is_host_login','is_guest_login']
continuous_cols = ['duration' ,'src_bytes', 'dst_bytes', 'wrong_fragment','urgent', 'hot',
'num_failed_logins', 'num_compromised', 'num_root' ,'num_file_creations',
'num_shells', 'num_access_files', 'num_outbound_cmds', 'count' ,'srv_count',
'serror_rate', 'srv_serror_rate' ,'rerror_rate' ,'srv_rerror_rate',
'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate' ,'dst_host_count',
'dst_host_srv_count' ,'dst_host_same_srv_rate' ,'dst_host_diff_srv_rate',
'dst_host_same_src_port_rate' ,'dst_host_srv_diff_host_rate',
'dst_host_serror_rate' ,'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
'dst_host_srv_rerror_rate']

# transformer
class OutcomeCreater(Transformer):
    def __init__(self):
        super().__init__()
    def _transform(self, dataset):
        label_to_binary = udf(lambda name: 0.0 if name == 'normal' else 1.0)
        output_df = dataset.withColumn('outcome', label_to_binary(col('class'))).drop("class")
        output_df = output_df.withColumn('outcome', col('outcome').cast(DoubleType()))
        output_df = output_df.drop('difficulty')
        return output_df

class FeatureTypeCaster(Transformer):
    def __init__(self):
        super().__init__()
    def _transform(self, dataset):
        output_df = dataset
        for c in binary_cols + continuous_cols:
            output_df = output_df.withColumn(c, col(c).cast(DoubleType()))
        return output_df

class ColumnDropper(Transformer):
    def __init__(self, columns_to_drop=None):
        super().__init__()
        self.columns_to_drop = columns_to_drop or []
    def _transform(self, dataset):
        output_df = dataset
        for c in self.columns_to_drop:
            output_df = output_df.drop(c)
        return output_df
    
train_raw = spark.read.csv('gs://dataproc-staging-us-central1-922748260413-wf6dfmrk/notebooks/jupyter/KDDTrain+.txt', header=False).toDF(*col_names)
test_raw = spark.read.csv('gs://dataproc-staging-us-central1-922748260413-wf6dfmrk/notebooks/jupyter/KDDTest+.txt',  header=False).toDF(*col_names)

DOS=['back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','mailbomb']
probing=['ipsweep','nmap','portsweep','satan','mscan','saint']
U2R=['buffer_overflow','loadmodule','perl','rootkit','ps','sqlattack','xterm']

def add_attack(df):
    c = lower(col("class"))
    return (df.withColumn(
              "attack",
              when(c=='normal','normal')
              .when(c.isin(DOS),   'DOS')
              .when(c.isin(probing), 'probing')
              .when(c.isin(U2R),   'U2R')
              .otherwise('R2L'))
           )

train_raw2 = add_attack(train_raw)
test_raw2 = add_attack(test_raw)
label_indexer = StringIndexer(inputCol="attack", outputCol="label", handleInvalid="keep")
label_model = label_indexer.fit(train_raw2)
train_idx = label_model.transform(train_raw2)
test_idx = label_model.transform(test_raw2)
def get_preprocess_pipeline():
    stage_typecaster = FeatureTypeCaster()
    nominal_id_cols = [x + "_index" for x in nominal_cols]
    nominal_oh_cols = [x + "_encoded" for x in nominal_cols]
    stage_nominal_indexer = StringIndexer(inputCols=nominal_cols, outputCols=nominal_id_cols, handleInvalid="keep")
    stage_nominal_onehot = OneHotEncoder(inputCols=nominal_id_cols, outputCols=nominal_oh_cols, handleInvalid="keep")
    feature_cols = continuous_cols + binary_cols + nominal_oh_cols
    corelated_cols_to_remove = ["dst_host_serror_rate","srv_serror_rate","dst_host_srv_serror_rate",
                                "srv_rerror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate"]
    feature_cols= [c for c in feature_cols if c not in corelated_cols_to_remove]
    stage_vec= VectorAssembler(inputCols=feature_cols, outputCol="vectorized_features", handleInvalid="keep")
    stage_scaler= StandardScaler(inputCol="vectorized_features", outputCol="features")
    stage_drop= ColumnDropper(columns_to_drop = nominal_cols + nominal_id_cols + nominal_oh_cols +
                                 binary_cols + continuous_cols + ['vectorized_features','difficulty'])
    return Pipeline(stages=[stage_typecaster, stage_nominal_indexer, stage_nominal_onehot,
                            stage_vec, stage_scaler, stage_drop])
feat_pipe = get_preprocess_pipeline()
feat_model = feat_pipe.fit(train_idx)
train_df = feat_model.transform(train_idx)
test_df = feat_model.transform(test_idx)

test_df.printSchema()
test_df.select("attack","label","features").show(5, truncate=False)

# Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train_df)
predictions_train_lr = lr_model.transform(train_df)
predictions_test_lr = lr_model.transform(test_df)

# Train accuracy
accuracy_train = (
    predictions_train_lr.filter(F.col("label") == F.col("prediction"))
    .count()
    / float(predictions_train_lr.count())
)

# Test accuracy
accuracy_test = (
    predictions_test_lr.filter(F.col("label") == F.col("prediction"))
    .count()
    / float(predictions_test_lr.count())
)

print(f"[Logistic Regression] Train Accuracy : {np.round(accuracy_train * 100, 2)}%")
print(f"[Logistic Regression] Test Accuracy  : {np.round(accuracy_test * 100, 2)}%")

### Q2-5 Output

Refer to q2-5_1, q2-5_2, q2-5_3, q2-5_4

## Q2-6

### Q2-6 code

In [ ]:
df = predictions_test_lr
num_partitions = df.rdd.getNumPartitions()
print(f"Number of partitions: {num_partitions}")

### Q2-6 Output

Refer to q2-6.png

# Q3

## Q3-1

In [35]:
import urllib.request
url = "https://www.guannanqu.com/files/cmutoolchain/sherlock.txt"
urllib.request.urlretrieve(url, "sherlock.txt")

('sherlock.txt', <http.client.HTTPMessage at 0x262ed201300>)

In [38]:
from pyspark import SparkContext
spark = SparkSession.builder.master("local[*]").appName("Q3-1").getOrCreate()
sc = spark.sparkContext
# define function
def count_target_words(rdd_lines, target_list):
    # converts all characters in the line string to lowercase and split the modfied string to a list of substrings
    words = rdd_lines.flatMap(lambda line: line.lower().split())

    # find the target words
    filtered = words.filter(lambda w: w in target_list)

    # map to (word,1)-(key,value)
    pairs = filtered.map(lambda w: (w, 1))

    # reduceByKey to sum counts
    counts = pairs.reduceByKey(lambda a, b: a + b)

    # collect final results
    result = counts.collect()
    return result

# load file
rdd_lines = sc.textFile("sherlock.txt")

# define target list
target_list = ["sherlock", "holmes", "watson", "lestrade", "moriarty"]

# apply function
result = count_target_words(rdd_lines, target_list)

# print result
print(result)

sc.stop()


[('holmes', 462), ('sherlock', 101), ('watson', 81), ('lestrade', 37)]


### Q3-1 Explaination

I define a function to do the count_word task, the function runs as follows:
-   flatMap: I split each text line into small words. It makes one long list of all words, so Spark can count them easily.
-   filter: I only keep the words we care about (like sherlock, holmes, etc.). This removes other useless words and saves time.
-   map: I turn every word into a pair (word, 1), which means “this word appears once.”
-   reduceByKey: Spark adds up all the 1’s for the same word.
-   collect: It brings counted result as the final result.

## Q3-2

In [42]:
spark = SparkSession.builder.master("local[*]").appName("Q3-2").getOrCreate()
sc = spark.sparkContext

def count_k_words(rdd_lines, k):

    # converts all characters in the line string to lowercase and split the modfied string to a list of substrings
    words = rdd_lines.flatMap(lambda line: line.lower().split())
    
    # (word, 1)
    pairs = words.map(lambda w: (w, 1))
    
    # reduceByKey to sum counts
    counts = pairs.reduceByKey(lambda a, b: a + b)
    
    # sort by count desc (key = count), then take top k
    sorted_counts = counts.sortBy(lambda kv: kv[1], ascending=False)
    topk = sorted_counts.take(k)
    return topk

rdd_lines = sc.textFile("sherlock.txt")
print(count_k_words(rdd_lines, 50))

sc.stop()

[('the', 5811), ('and', 3066), ('i', 2994), ('of', 2779), ('to', 2762), ('a', 2680), ('in', 1818), ('that', 1750), ('it', 1710), ('you', 1548), ('he', 1465), ('was', 1410), ('his', 1159), ('is', 1142), ('my', 1006), ('have', 929), ('with', 877), ('as', 861), ('had', 830), ('at', 780), ('which', 776), ('for', 751), ('not', 663), ('but', 651), ('be', 644), ('me', 634), ('we', 532), ('this', 531), ('from', 511), ('there', 502), ('said', 486), ('upon', 466), ('holmes', 462), ('so', 447), ('him', 433), ('her', 429), ('she', 426), ('your', 404), ('all', 403), ('very', 401), ('no', 398), ('been', 393), ('on', 391), ('what', 387), ('by', 373), ('one', 370), ('then', 365), ('are', 357), ('were', 349), ('an', 338)]


### Q3-2 Explaination

I define a function to find the most frequent words in the text. The function works as follows:

-   flatMap: I split each line into small lowercase words using regex. It creates one long list of all words, making counting easier.
-   map: I turn every word into a pair (word, 1), meaning “this word appears once.”
-   reduceByKey: Spark adds all the 1’s for the same word to get the total count.
-   sortBy: I sort all (word, count) pairs by the count in descending order, so the most common words come first.
-   take: I take only the top k(50) words as the final output.